In [0]:
%run ./01-ATSConfigs

In [0]:
%run ./03-Endpoints

In [0]:
%run ./04-VectorSearch

In [0]:
# from mlflow.deployments import get_deploy_client
# import time

# client = get_deploy_client("databricks")

# start = time.time()
# try:
#     resp = client.predict(
#         endpoint="databricks-gte-large-en",     # control test (Databricks native)
#         inputs={"input": ["hello"]}
#     )
#     print("OK (databricks-gte-large-en)")
#     print(resp)
# except Exception as e:
#     print("FAILED (databricks-gte-large-en)")
#     print(e)

# print("Elapsed seconds:", int(time.time() - start))


In [0]:
# Databricks notebook source
# MAGIC %run ./01-ATSConfigs

# COMMAND ----------

# MAGIC %run ./03-Endpoints

# COMMAND ----------

# MAGIC %run ./04-VectorSearch

# COMMAND ----------

class Setup:
    def __init__(self):
        pass

    def setup_metadata(self):
        print(f"Creating Metadata for Jobs...", end="")
        spark.sql(
            f"""create table if not exists {conf.jobs_metadata_table_name}(
                            job_name string,
                            last_load_date date,
                            execution_time timestamp,
                            description string)
                  """
        )
        spark.sql(f"truncate table {conf.jobs_metadata_table_name}")
        spark.sql(
            f"""insert into {conf.jobs_metadata_table_name} 
                      values('{conf.profile_ingestion_job_name}',
                             '{conf.profile_last_ingestion_date}',
                             current_timestamp(),
                             'Initial Configuration')"""
        )
        spark.sql(
            f"""insert into {conf.jobs_metadata_table_name} 
                      values('{conf.profile_bronze_job_name}',
                             '{conf.profile_last_bronze_load_date}',
                             current_timestamp(),
                             'Initial Configuration')"""
        )
        spark.sql(
            f"""insert into {conf.jobs_metadata_table_name} 
                      values('{conf.profile_silver_job_name}',
                             '{conf.profile_last_silver_load_date}',
                             current_timestamp(),
                             'Initial Configuration')"""
        )
        spark.sql(
            f"""insert into {conf.jobs_metadata_table_name} 
                      values('{conf.index_sync_job_name}',
                             '{conf.last_index_sync_date}',
                             current_timestamp(),
                             'Initial Configuration')"""
        )
        spark.sql(
            f"""insert into {conf.jobs_metadata_table_name} 
                      values('{conf.jd_ingestion_job_name}',
                             '{conf.jd_last_ingestion_date}',
                             current_timestamp(),
                             'Initial Configuration')"""
        )
        spark.sql(
            f"""insert into {conf.jobs_metadata_table_name} 
                      values('{conf.jd_bronze_job_name}',
                             '{conf.jd_last_bronze_load_date}',
                             current_timestamp(),
                             'Initial Configuration')"""
        )
        spark.sql(
            f"""insert into {conf.jobs_metadata_table_name} 
                      values('{conf.jd_silver_job_name}',
                             '{conf.jd_last_silver_load_date}',
                             current_timestamp(),
                             'Initial Configuration')"""
        )
        spark.sql(
            f"""insert into {conf.jobs_metadata_table_name} 
                      values('{conf.jd_profile_job_name}',
                             '{conf.jd_profile_last_load_date}',
                             current_timestamp(),
                             'Initial Configuration')"""
        )
        print("Done")

    def setup_landing_zone(self):
        print(f"Creating Landing Zone directories...", end="")
        profile_landing_zone = (
            f"/Volumes/{conf.catalog}/{conf.db}/{conf.profile_landing_zone}"
        )
        dbutils.fs.mkdirs(profile_landing_zone)
        jd_landing_zone = f"/Volumes/{conf.catalog}/{conf.db}/{conf.jd_landing_zone}"
        dbutils.fs.mkdirs(jd_landing_zone)
        print("Done")

    def setup_tables(self):
        print(f"Creating tables...", end="")
        spark.sql(
            f"""create table if not exists {conf.profile_bronze_table_name}(
                      id BIGINT GENERATED ALWAYS AS IDENTITY,
                      path string,
                      modificationTime timestamp,
                      length bigint,
                      content binary
                    )TBLPROPERTIES (delta.enableChangeDataFeed = true)"""
        )

        spark.sql(
            f"""create table if not exists {conf.profile_silver_table_name}_stg(
                      source string,
                      text_content string)"""
        )

        spark.sql(
            f"""create table if not exists {conf.profile_silver_table_name}(
                      id BIGINT GENERATED ALWAYS AS IDENTITY,
                      source string,
                      text_content string,
                      json_content string
                    )TBLPROPERTIES (delta.enableChangeDataFeed = true)"""
        )

        spark.sql(
            f"""create table if not exists {conf.jd_bronze_table_name}(
                      id BIGINT GENERATED ALWAYS AS IDENTITY,
                      path string,
                      modificationTime timestamp,
                      length bigint,
                      content binary
                    )TBLPROPERTIES (delta.enableChangeDataFeed = true)"""
        )

        spark.sql(
            f"""create table if not exists {conf.jd_silver_table_name}_stg(
                      source string,
                      text_content string)"""
        )

        spark.sql(
            f"""create table if not exists {conf.jd_silver_table_name}(
                      id BIGINT GENERATED ALWAYS AS IDENTITY,
                      source string,
                      text_content string,
                      json_content string
                    )TBLPROPERTIES (delta.enableChangeDataFeed = true)"""
        )

        spark.sql(
            f"""create table if not exists {conf.jd_profile_table_name}_stg(
                      jd_id bigint,
                      profile_id bigint)"""
        )

        spark.sql(
            f"""create table if not exists {conf.jd_profile_table_name}(
                      id BIGINT GENERATED ALWAYS AS IDENTITY,
                      jd_id	bigint,
                      jd_source	string,
                      jd_extract string,
                      profile_id bigint,
                      profile_source string,
                      profile_extract string,
                      ats_summary string,
                      generated_date timestamp
                    )TBLPROPERTIES (delta.enableChangeDataFeed = true)"""
        )
        print("Done")

    def setup_endpoints(self):
        ep = Endpoints()
        ep.create(name=conf.llm_endpoint_for_chat, model=conf.llm_chat_model_name)
        # ep.create(
        #     name=conf.embedding_model_endpoint_name,
        #     model=conf.embedding_model_name,
        #     task="embeddings",
        # )

    def setup_vector_index(self):
        vs = VectorSearch()
        vs.create_endpoint(endpoint_name=conf.vector_store_endpoint_name)

        vs.create_index(
            index_name=conf.vector_index_name,
            source_table_name=conf.profile_silver_index_source_table,
            primary_key="id",
            embedding_source_column="json_content",
            embedding_model_endpoint_name= conf.embedding_model_endpoint_name,
            vector_endpoint_name=conf.vector_store_endpoint_name,
        )

    def run(self):
        # Setup metadata if it was not already setup
        if (
            spark.sql("show tables")
            .where(
                f"database=='{conf.db}' and tableName=='{conf.jobs_metadata_table_name}'"
            )
            .count()
        ) == 0:
            self.setup_metadata()
        elif (spark.sql(f"select * from {conf.jobs_metadata_table_name}").count()) < 8:
            self.setup_metadata()

        # Setup landing zone directories
        self.setup_landing_zone()

        # Setup required tables
        self.setup_tables()

        # Setup required endpoints
        self.setup_endpoints()

        # Setup vector index endpoint and index
        self.setup_vector_index()

    def assert_table(self, table_name):
        assert spark.sql(f"SHOW TABLES IN {conf.catalog}.{conf.db}") \
                   .filter(f"isTemporary == false and tableName == '{table_name}'") \
                   .count() == 1, f"The table {table_name} is missing"
        print(f"Found {table_name} table in {conf.catalog}.{conf.db}: Success")

    def assert_count(self, table_name, expected_count):
        print(f"Validating record counts in {table_name}...", end='')
        actual_count = spark.read.table(f"{conf.catalog}.{conf.db}.{table_name}").count()
        assert actual_count == expected_count, f"Expected {expected_count:,} records, found {actual_count:,} in {table_name}" 
        print(f"Found {actual_count:,} / Expected {expected_count:,} records: Success")

    def assert_endpoint(self, endpoint_name):
        from mlflow.deployments import get_deploy_client
        dp_client = get_deploy_client("databricks")
        ep_list = dp_client.list_endpoints()
        ep_names = [ep["name"] for ep in ep_list]
        assert endpoint_name in ep_names, f"The endpoint '{endpoint_name}' is missing"
        print(f"Found {endpoint_name}: Success")

    def assert_vector_index(self, endpoint_name, index_name):
        # from databricks.vector_search import VectorSearchClient
        from databricks.vector_search.client import VectorSearchClient

        vs_client = VectorSearchClient(disable_notice=True)
        eps = vs_client.list_endpoints()
        ep_names = [ep["name"] for ep in eps["endpoints"]]
        assert endpoint_name in ep_names, f"The vector endpoint '{endpoint_name}' is missing"
        print(f"Found {endpoint_name}: Success")
        idxs = vs_client.list_indexes(endpoint_name)
        idx_names = [idx["name"] for idx in idxs["vector_indexes"]]
        assert index_name in idx_names, f"The vector index '{index_name}' is missing"
        print(f"Found {index_name}: Success")

    def assert_dir(self, dir_name):
        dirs = dbutils.fs.ls(f"/Volumes/{conf.catalog}/{conf.db}/{conf.landing_volume}")
        dir_list = [dir.path for dir in dirs]
        dir_path = f"dbfs:/Volumes/{conf.catalog}/{conf.db}/{conf.landing_volume}/{dir_name}/"
        assert dir_path in dir_list, f"The directory {dir_path} is missing"
        print(f"Found {dir_path}: Success") 
        
    def validate(self):
        import time
        start = int(time.time())
        print(f"\nStarting setup validation ...")
        assert spark.sql(f"SHOW DATABASES IN {conf.catalog}") \
                    .filter(f"databaseName == '{conf.db}'") \
                    .count() == 1, f"The database '{conf.catalog}.{conf.db}' is missing"
        print(f"Found database {conf.catalog}.{conf.db}: Success")

        print(f"\nStarting table validation...")
        self.assert_table(conf.jobs_metadata_table_name)
        self.assert_table(conf.profile_bronze_table_name)
        self.assert_table(conf.profile_silver_table_name)
        self.assert_table(conf.jd_bronze_table_name)
        self.assert_table(conf.jd_silver_table_name)
        self.assert_table(conf.jd_profile_table_name)

        print(f"\nStarting landing zone validation...")
        self.assert_dir(conf.profile_source)
        self.assert_dir(conf.jd_source) 

        print(f"\nStarting metadata validation...")
        self.assert_count(conf.jobs_metadata_table_name, 8)        

        print(f"\nStarting endpoint validation...")
        self.assert_endpoint(conf.llm_endpoint_for_chat)
        self.assert_endpoint(conf.embedding_model_endpoint_name)
        self.assert_vector_index(conf.vector_store_endpoint_name, conf.vector_index_name)
        print(f"Setup validation completed in {int(time.time()) - start} seconds") 

# COMMAND ----------

setup = Setup()
setup.run()
setup.validate()